In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from src.data.data_preprocess import masterdata

ImportError: cannot import name 'masterdata' from 'src.data.data_preprocess' (C:\Users\danil\Desktop\PROJECTS\formula1\src\data\data_preprocess.py)

In [3]:
masterdata.head()

,resultId,raceId,driverId,constructorId_x,number_x,grid,race_position,positionText,positionOrder,points,...,constructor_name,nationality_y,url_constructor,qualifyId,constructorId_y,qualifying_position,q1,q2,q3,top3
0,1,18,1,1,22,1,1.0,1,1,10.0,...,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1.0,1.0,1.0,1:26.572,1:25.187,1:26.714,1
1,2,18,2,2,3,5,2.0,2,2,8.0,...,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber,5.0,2.0,5.0,1:25.960,1:25.518,1:27.236,1
2,3,18,3,3,7,7,3.0,3,3,6.0,...,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...,7.0,3.0,7.0,1:26.295,1:26.059,1:28.687,1
3,4,18,4,4,5,11,4.0,4,4,5.0,...,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...,12.0,4.0,12.0,1:26.907,1:26.188,\N,0
4,5,18,5,1,23,3,5.0,5,5,4.0,...,McLaren,British,http://en.wikipedia.org/wiki/McLaren,3.0,1.0,3.0,1:25.664,1:25.452,1:27.079,0


In [4]:
import requests
from bs4 import BeautifulSoup
import re
from tqdm import tqdm
tqdm.pandas()


gp_table = masterdata[['raceId', 'year', 'url_gp']].drop_duplicates().sort_values(by='year')
gp_table = gp_table[gp_table >=2010]

,raceId,year,url_gp
20149,839,1950,http://en.wikipedia.org/wiki/1950_Italian_Gran...
20131,838,1950,http://en.wikipedia.org/wiki/1950_French_Grand...
20117,837,1950,http://en.wikipedia.org/wiki/1950_Belgian_Gran...
20099,836,1950,http://en.wikipedia.org/wiki/1950_Swiss_Grand_...
20066,835,1950,http://en.wikipedia.org/wiki/1950_Indianapolis...
...,...,...,...
26320,1123,2024,https://en.wikipedia.org/wiki/2024_Australian_...
26300,1122,2024,https://en.wikipedia.org/wiki/2024_Saudi_Arabi...
26280,1121,2024,https://en.wikipedia.org/wiki/2024_Bahrain_Gra...
26479,1131,2024,https://en.wikipedia.org/wiki/2024_Austrian_Gr...


In [5]:
def get_weather_text_from_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        infobox = soup.find("table", class_="infobox")
        if not infobox:
            return None

        for row in infobox.find_all("tr"):
            header = row.find("th")
            if header and 'weather' in header.text.lower():
                cell = row.find("td")
                return cell.text.strip() if cell else None

        return None

    except Exception as e:
        print(f"Error for {url}: {e}")
        return None

In [6]:
gp_table

,raceId,year,url_gp
20149,839,1950,http://en.wikipedia.org/wiki/1950_Italian_Gran...
20131,838,1950,http://en.wikipedia.org/wiki/1950_French_Grand...
20117,837,1950,http://en.wikipedia.org/wiki/1950_Belgian_Gran...
20099,836,1950,http://en.wikipedia.org/wiki/1950_Swiss_Grand_...
20066,835,1950,http://en.wikipedia.org/wiki/1950_Indianapolis...
...,...,...,...
26320,1123,2024,https://en.wikipedia.org/wiki/2024_Australian_...
26300,1122,2024,https://en.wikipedia.org/wiki/2024_Saudi_Arabi...
26280,1121,2024,https://en.wikipedia.org/wiki/2024_Bahrain_Gra...
26479,1131,2024,https://en.wikipedia.org/wiki/2024_Austrian_Gr...


In [7]:
from tqdm import tqdm
tqdm.pandas()

gp_table['weather_raw'] = gp_table['url_gp'].progress_apply(get_weather_text_from_url)


100%|██████████████████████████████████████████████████████████████████████████████| 1125/1125 [09:23<00:00,  2.00it/s]


In [15]:
gp_table.to_csv("C:/Users/danil/Desktop/PROJECTS/formula1/datasets/processed/gp_weather.csv", index=False)


In [12]:
gp_table.to_csv("C:/Users/danil/Desktop/PROJECTS/formula1/datasets/processed/gp_weather.csv", index=False)